Train Monthly Model - Live Production Data
Trains Prophet on monthly gold data, evaluates against baseline, saves 3-month forecast.

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_gold
import pandas as pd
from prophet import Prophet

blob_service = get_blob_service(storage_account_name, storage_account_key)

FORECAST_BASE = "live/battery"

def wape(y_true, y_pred):
    return abs(y_true - y_pred).sum() / abs(y_true).sum()

def compute_working_days(period_start, period_end, holiday_dates_set):
    all_days = pd.date_range(period_start, period_end)
    return sum(1 for d in all_days if d.weekday() != 6 and d.date() not in holiday_dates_set)

Load gold + holiday calendar, build Prophet-ready rate series

In [0]:
gold_monthly = read_gold(blob_service, f"{FORECAST_BASE}/data/phase1_overall_monthly_live.parquet")
gold_monthly["month_start"] = pd.to_datetime(gold_monthly["month_start"])

holiday_calendar = read_gold(blob_service, f"{FORECAST_BASE}/data/reference/holiday_calendar.parquet")
holiday_calendar["date"] = pd.to_datetime(holiday_calendar["date"])
holiday_dates_set = set(holiday_calendar["date"].dt.date)

monthly_rate_df = gold_monthly[["month_start", "units_per_working_day"]].rename(
    columns={"month_start": "ds", "units_per_working_day": "y"}
)
print(monthly_rate_df.tail(10))

Backtest, comparing actual totals

In [0]:
train_monthly = monthly_rate_df.iloc[:-3]
test_monthly = monthly_rate_df.iloc[-3:]

test_actual_totals = gold_monthly.iloc[-3:]["total_units_sold"].values
test_working_days = gold_monthly.iloc[-3:]["working_days"].values

m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False, growth='flat')
m_test.fit(train_monthly)

future_test = m_test.make_future_dataframe(periods=3, freq="MS")
forecast_test = m_test.predict(future_test)
test_rate_preds = forecast_test.tail(3)["yhat"].clip(lower=0).values

test_total_preds = test_rate_preds * test_working_days

print(f"Rate-based Monthly Prophet WAPE: {wape(test_actual_totals, test_total_preds):.3%}")

naive_rate = train_monthly["y"].tail(3).mean()
naive_totals = naive_rate * test_working_days
print(f"Rate-based Naive Baseline WAPE: {wape(test_actual_totals, naive_totals):.3%}")

Final 3-month forecast, retrained on all data

In [0]:
m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False, growth='flat')
m_final.fit(monthly_rate_df)

future_final = m_final.make_future_dataframe(periods=3, freq="MS")
forecast_final = m_final.predict(future_final)
forecast_final[["yhat", "yhat_lower", "yhat_upper"]] = forecast_final[["yhat", "yhat_lower", "yhat_upper"]].clip(lower=0)

next_3_months = forecast_final.tail(3).copy()

# Compute actual working days for each forecasted month
next_3_months["working_days"] = next_3_months["ds"].apply(
    lambda month_start: max(compute_working_days(month_start, month_start + pd.offsets.MonthEnd(0), holiday_dates_set), 1)
)

next_3_months["predicted_total"] = next_3_months["yhat"] * next_3_months["working_days"]
next_3_months["predicted_total_lower"] = next_3_months["yhat_lower"] * next_3_months["working_days"]
next_3_months["predicted_total_upper"] = next_3_months["yhat_upper"] * next_3_months["working_days"]

print(next_3_months[["ds", "working_days", "predicted_total", "predicted_total_lower", "predicted_total_upper"]])